# UrbanRay AI

Upload the curated project ZIP or the repository ZIP downloaded from GitHub. It contains the current 88-building model, trained simulator surrogate, Landsat exports and exploratory summaries. Large generated training tables are excluded.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io
uploaded = files.upload()
choices = [name for name in uploaded if name.endswith('.zip')]
if len(choices) != 1:
    raise ValueError('Upload one UrbanRay project ZIP')
base = Path('/content/urbanray_upload')
base.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[choices[0]])) as z:
    for member in z.infolist():
        if not (base / member.filename).resolve().is_relative_to(base.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(base)
roots = list(base.rglob('UrbanRay_Run_10Date_Validation.py'))
if len(roots) != 1:
    raise ValueError('Expected one current project in the archive')
project = roots[0].parent
print('Project:', project)
print('Landsat observation rows:', sum(1 for _ in open(project/'urbanray_qualified_landsat_pixels_2025_2026.csv'))-1)


In [ ]:
%cd {project}
!python -m pip install -q -r requirements-tested.txt


In [ ]:
!python -m pytest -q


## View the already-completed ten-scene results

The 2026 numbers below compare a modeled emission-temperature proxy to satellite retrieved land-surface temperature. This is exploratory and does **not** establish individual-rooftop accuracy.

In [ ]:
import pandas as pd, json
from IPython.display import display
scenes = pd.read_csv('outputs/multidate_validation/scene_level_EXPLORATORY.csv')
metrics = json.load(open('outputs/multidate_validation/heldout/heldout_diagnostic.json'))
display(scenes)
print('Held-out raw metrics:', metrics['heldout_raw'])
print('Held-out bias-corrected metrics:', metrics['heldout_bias_corrected'])

In [ ]:
from IPython.display import Image, display
display(Image(filename='outputs/multidate_validation/audit/heldout_scene_comparison.png'))

## OPTIONAL: Plan or rerun selected dates

The 10-date outputs are already archived as summaries. Rerunning requires downloading historical Open-Meteo weather and significant compute. The original full weather/per-date simulation products were not included in the uploaded summaries.

In [ ]:
!python UrbanRay_Run_10Date_Validation.py --all --dry-run

To rerun one date, remove the leading `#` in the next cell's Python source and run it. To rerun all, use `--all`; this can be costly in Colab.

In [ ]:
# !python UrbanRay_Run_10Date_Validation.py --dates 2025-04-23

## OPTIONAL: Retrain the AI

This targets simulated multi-bounce shortwave absorption. It is **not** training a measured-temperature prediction model.
First generate a fresh multi-day face table using the README weather and simulation commands. The original large training CSV is excluded; exact historical retraining requires the original archive.


In [ ]:
# !python train_surrogate.py --faces outputs/new_historical_run/physics_training_faces.csv --out outputs/ml_retrained
